In [1]:
import pandas as pd

from langchain import PromptTemplate
from langchain.chains import LLMChain
from langchain.chat_models import ChatOpenAI

from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(temperature=0, model_name='gpt-4', request_timeout=120)

In [2]:
df = pd.read_csv("../data/justification-finetune-data/skeptic-205-records.csv")

In [3]:
df.head()

,video_id,title,author,description,org_transcript,eng_transcript,image_base64,url,Summary_Claims,Justification
0,dZ7xeVCYC5M,How I Would Invest $1000 If I Were In My 20s,The Game w/ Alex Hormozi,My new book $100M Leads is now LIVE. Grab your...,i [ __ ] guarantee you that you will be makin...,"I cannot guarantee you that, but I can assure ...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBAgKDQoICg...,https://www.youtube.com/watch?v=dZ7xeVCYC5M\n,The influencer claims that investing in self-e...,The influencer's claim that investing in self-...
1,TK74cx0p-NM,Nestle India share price jumps by over 1.5% af...,5paisa,Nestle India share price jumped more than 1.5%...,[Music] hi everyone 1.5 Financial results Mar...,[Music] Hi everyone! The financial results for...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBgcICAgIBw...,https://www.youtube.com/watch?v=TK74cx0p-NM,The financial influencer claims that the finan...,The claims made by the influencer are factual ...
2,JnIYdowe9KE,Swing Trading Profit Double - Best Strategy,Stock Learners,In This Video I Have Share One Of My Trade Log...,तो स्टॉक मार्केट में कैसे आप स्विंग ट्रेडिंग क...,So how can you make a good return by swing tra...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBBAQDxAQEB...,https://www.youtube.com/watch?v=JnIYdowe9KE\n,The financial influencer claims that by swing ...,The influencer's claim that one can make signi...
3,GiiHU87xuGY,Top 3 positive stocks | Stocks for 23-Oct-2023...,PM Stock Academy,NaN,"वयलकम दोस्तो, बीयम स्टॉक अकडमी के एक नए वीडियो...","""Hello friends, welcome to the video of the Ve...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBAgICAgICA...,https://www.youtube.com/watch?v=GiiHU87xuGY\n,The financial influencer suggests three stocks...,The claims made by the influencer are based on...
4,gsXgM6WLJsg,"Turning $100 Into $1,000 Trading Stocks | Ep.1",Jenny Hoyos,get up to 10 FREE stocks (deposit at least $10...,this is a penny and last week i tried turning...,"""This is a penny, and last week I attempted to...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBg0KCQkJCQ...,https://www.youtube.com/watch?v=gsXgM6WLJsg\n,The financial influencer claims that he is att...,"The claim of turning 100 into 1,000 through st..."


In [12]:
def assign_gt_labels(sample):
   summary_claims = sample["Summary_Claims"]
   justification = sample["Justification"]

   template="""
      You are a Financial Analyst. You are provided with Claims made by Financial Infuencer. \
      Your task is to classify if the Claims are true or false based on provided Justification. If there are no Claims then assign neutral. \
      Restrict the response to one of the label categories. No need of any explanation.

      Claim Summary: {summary_claims}
      Justifications: {justification}

      Category: 
      """
   prompt = PromptTemplate(
      input_variables=['summary_claims', 'justification'],
      template=template
   )
   chain = LLMChain(llm=llm, prompt=prompt)
   labels = chain.run({'summary_claims':summary_claims, 'justification':justification})
   # formatted_output = parser.parse(labels)

   return labels

In [13]:
df["labels"] = df.apply(assign_gt_labels, axis=1)

In [14]:
df.head()

,video_id,title,author,description,org_transcript,eng_transcript,image_base64,url,Summary_Claims,Justification,labels
0,dZ7xeVCYC5M,How I Would Invest $1000 If I Were In My 20s,The Game w/ Alex Hormozi,My new book $100M Leads is now LIVE. Grab your...,i [ __ ] guarantee you that you will be makin...,"I cannot guarantee you that, but I can assure ...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBAgKDQoICg...,https://www.youtube.com/watch?v=dZ7xeVCYC5M\n,The influencer claims that investing in self-e...,The influencer's claim that investing in self-...,True
1,TK74cx0p-NM,Nestle India share price jumps by over 1.5% af...,5paisa,Nestle India share price jumped more than 1.5%...,[Music] hi everyone 1.5 Financial results Mar...,[Music] Hi everyone! The financial results for...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBgcICAgIBw...,https://www.youtube.com/watch?v=TK74cx0p-NM,The financial influencer claims that the finan...,The claims made by the influencer are factual ...,Neutral
2,JnIYdowe9KE,Swing Trading Profit Double - Best Strategy,Stock Learners,In This Video I Have Share One Of My Trade Log...,तो स्टॉक मार्केट में कैसे आप स्विंग ट्रेडिंग क...,So how can you make a good return by swing tra...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBBAQDxAQEB...,https://www.youtube.com/watch?v=JnIYdowe9KE\n,The financial influencer claims that by swing ...,The influencer's claim that one can make signi...,True
3,GiiHU87xuGY,Top 3 positive stocks | Stocks for 23-Oct-2023...,PM Stock Academy,NaN,"वयलकम दोस्तो, बीयम स्टॉक अकडमी के एक नए वीडियो...","""Hello friends, welcome to the video of the Ve...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBAgICAgICA...,https://www.youtube.com/watch?v=GiiHU87xuGY\n,The financial influencer suggests three stocks...,The claims made by the influencer are based on...,False
4,gsXgM6WLJsg,"Turning $100 Into $1,000 Trading Stocks | Ep.1",Jenny Hoyos,get up to 10 FREE stocks (deposit at least $10...,this is a penny and last week i tried turning...,"""This is a penny, and last week I attempted to...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBg0KCQkJCQ...,https://www.youtube.com/watch?v=gsXgM6WLJsg\n,The financial influencer claims that he is att...,"The claim of turning 100 into 1,000 through st...",True


In [23]:
df.to_csv("../data/assigned-labels-205.csv", index=False)

In [20]:
df.labels.value_counts()

labels
True       114
False       48
Neutral     43
Name: count, dtype: int64

In [24]:
df.drop(columns=["image_base64"])

,video_id,title,author,description,org_transcript,eng_transcript,url,Summary_Claims,Justification,labels
0,dZ7xeVCYC5M,How I Would Invest $1000 If I Were In My 20s,The Game w/ Alex Hormozi,My new book $100M Leads is now LIVE. Grab your...,i [ __ ] guarantee you that you will be makin...,"I cannot guarantee you that, but I can assure ...",https://www.youtube.com/watch?v=dZ7xeVCYC5M\n,The influencer claims that investing in self-e...,The influencer's claim that investing in self-...,True
1,TK74cx0p-NM,Nestle India share price jumps by over 1.5% af...,5paisa,Nestle India share price jumped more than 1.5%...,[Music] hi everyone 1.5 Financial results Mar...,[Music] Hi everyone! The financial results for...,https://www.youtube.com/watch?v=TK74cx0p-NM,The financial influencer claims that the finan...,The claims made by the influencer are factual ...,Neutral
2,JnIYdowe9KE,Swing Trading Profit Double - Best Strategy,Stock Learners,In This Video I Have Share One Of My Trade Log...,तो स्टॉक मार्केट में कैसे आप स्विंग ट्रेडिंग क...,So how can you make a good return by swing tra...,https://www.youtube.com/watch?v=JnIYdowe9KE\n,The financial influencer claims that by swing ...,The influencer's claim that one can make signi...,True
3,GiiHU87xuGY,Top 3 positive stocks | Stocks for 23-Oct-2023...,PM Stock Academy,NaN,"वयलकम दोस्तो, बीयम स्टॉक अकडमी के एक नए वीडियो...","""Hello friends, welcome to the video of the Ve...",https://www.youtube.com/watch?v=GiiHU87xuGY\n,The financial influencer suggests three stocks...,The claims made by the influencer are based on...,False
4,gsXgM6WLJsg,"Turning $100 Into $1,000 Trading Stocks | Ep.1",Jenny Hoyos,get up to 10 FREE stocks (deposit at least $10...,this is a penny and last week i tried turning...,"""This is a penny, and last week I attempted to...",https://www.youtube.com/watch?v=gsXgM6WLJsg\n,The financial influencer claims that he is att...,"The claim of turning 100 into 1,000 through st...",True
...,...,...,...,...,...,...,...,...,...,...
200,24GdZGDANmU,"Vodafone Idea Share Price climbed over 7.2%, K...",5paisa,"On 11-Oct-23, Vodafone Idea (VI) share price i...",आज वोडाफोन आइडिया की शेर्ज में 7.20% से जादा क...,"Today, the share of Vodafone Idea has increase...",https://www.youtube.com/watch?v=24GdZGDANmU,The financial influencer claims that the share...,The claim about the increase in Vodafone Idea'...,True
201,K1Kv5YloQ7Q,WHY YOU ARE MAKING LOSSES? || HOW TO EARN MONE...,Amrev,WHY YOU ARE MAKING LOSSES? || HOW TO EARN MONE...,"नार्मल इनसान जॉब करता होता होगा, कॉलेज़ पर जात...","A normal person might have a job, go to colleg...",https://www.youtube.com/watch?v=K1Kv5YloQ7Q,No claims made,No claims to analyse,Neutral
202,W3TLPwrGvqw,How to start trading || Minimum capital,Amrev,How to start trading || Minimum capital\r\n\r\...,मिनीमम देखे यहाँ पर दस रुपए से आप चालू कर सकते...,You can start with a minimum of ten rupees her...,https://www.youtube.com/watch?v=W3TLPwrGvqw,The influencer claims that one can start inves...,The claim that starting with a small investmen...,True
203,4wx7ZoNa6bA,I Make a Living Day Trading This One Simple St...,Scarface Trades,"In this video, I explain my full time day trad...",I've been training for over five years and I'...,"""I have been training for over five years and ...",https://www.youtube.com/watch?v=4wx7ZoNa6bA\n,The financial influencer claims that a success...,The claims made by the influencer are generall...,True
